# Colab Benchmarking Starter Lab

## Setup

In [ ]:
import statistics
import time
import torch

print('CUDA available:', torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

## Theory-to-code bridge
GPU kernels are asynchronous, so synchronization is required for accurate timing.

In [ ]:
def benchmark(fn, repeats=10, warmups=3):
    for _ in range(warmups):
        fn()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    timings = []
    for _ in range(repeats):
        start = time.perf_counter()
        fn()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        timings.append(time.perf_counter() - start)
    return statistics.median(timings), statistics.mean(timings)

## Benchmark or experiment

In [ ]:
size = 10_000_000
cpu_a = torch.randn(size)
cpu_b = torch.randn(size)

def cpu_add():
    _ = cpu_a + cpu_b

cpu_median, cpu_mean = benchmark(cpu_add)
print('CPU median:', cpu_median, 'mean:', cpu_mean)

if torch.cuda.is_available():
    gpu_a = cpu_a.to('cuda')
    gpu_b = cpu_b.to('cuda')
    def gpu_add():
        _ = gpu_a + gpu_b
    gpu_median, gpu_mean = benchmark(gpu_add)
    print('GPU median:', gpu_median, 'mean:', gpu_mean)
    print('Speedup (CPU median / GPU median):', cpu_median / gpu_median)


## Interpretation
Discuss whether speedup is meaningful for this operation size and why.

## Extensions
- Change tensor size
- Compare addition vs matrix multiplication